In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import pingouin as pg
import sys
import textwrap



# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Matplotlib": plt.matplotlib.__version__,
    "Seaborn": sns.__version__,
    "Statsmodels": sm.__version__,
    "Pyarrow": pa.__version__,
    "Pingouin": pg.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
df_versions

,Library,Version
0,Python,3.13.9
1,Pandas,2.3.3
2,NumPy,2.3.4
3,Matplotlib,3.10.7
4,Seaborn,0.13.2
5,Statsmodels,0.14.5
6,Pyarrow,22.0.0
7,Pingouin,0.5.5


## Read Data

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# From Arrow IPC (preferred)
table_loaded = ipc.open_file('../Data/crime_data.arrow').read_all()
df = table_loaded.to_pandas(types_mapper=pd.ArrowDtype)

In [4]:
df.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,year,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,arrest_date,race,fbi_code_desc,month,day_of_week,quarter,year_quarter,time_of_day
0,JJ532138,2025-12-23 10:45:00,028XX W TAYLOR ST,2024,NARCOTICS,POSSESS - HEROIN (WHITE),N,NARCOTICS,POSSESS - HEROIN (WHITE),ALLEY,True,False,1135,011,28,27,2025,18,60612,106718949.397,Garfield Park,GARFIELD PARK,89976069.5947,011,3,1135,2025-12-23 10:45:00,WHITE,Drug Abuse Violations,December,Tuesday,Q4,2025-Q4,Morning
1,JJ530701,2025-12-22 07:03:00,007XX N FRANKLIN ST,1330,CRIMINAL TRESPASS,TO LAND,N,CRIMINAL TRESPASS,TO LAND,SMALL RETAIL STORE,True,False,1831,018,42,8,2025,26,60654,15869961.5669,River North,RIVER NORTH,38766442.5194,018,3,1831,2025-12-22 07:12:00,BLACK,Miscellaneous Non-Index Offenses,December,Monday,Q4,2025-Q4,Early Morning
2,JJ518020,2025-12-10 15:20:00,031XX S ASHLAND AVE,0460,BATTERY,SIMPLE,N,BATTERY,SIMPLE,SMALL RETAIL STORE,True,False,912,009,12,59,2025,08B,60608,176505462.842,Mckinley Park,"BRIGHTON PARK,MCKINLEY PARK",39431799.6479,009,1,912,2025-12-10 17:32:00,BLACK,Simple Battery,December,Wednesday,Q4,2025-Q4,Afternoon
3,JJ496899,2025-11-21 04:50:00,031XX W HARRISON ST,0454,BATTERY,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",N,BATTERY,"AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...",POLICE FACILITY / VEHICLE PARKING LOT,True,False,1134,011,24,27,2025,08B,60612,106718949.397,Garfield Park,GARFIELD PARK,89976069.5947,011,3,1134,2025-11-21 04:15:00,BLACK,Simple Battery,November,Friday,Q4,2025-Q4,Early Morning
4,JJ492148,2025-11-17 06:20:00,035XX N CICERO AVE,0860,THEFT,RETAIL THEFT,I,THEFT,RETAIL THEFT,CONVENIENCE STORE,True,False,1731,017,30,15,2025,06,60641,113903341.241,Portage Park,PORTAGE PARK,110196097.139,017,3,1731,2025-11-17 06:30:00,BLACK,Larceny – Theft,November,Monday,Q4,2025-Q4,Early Morning


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8486097 entries, 0 to 8486096
Data columns (total 34 columns):
 #   Column                  Dtype                
---  ------                  -----                
 0   case_number             string[pyarrow]      
 1   date                    timestamp[s][pyarrow]
 2   block                   string[pyarrow]      
 3   iucr                    string[pyarrow]      
 4   primary_description     string[pyarrow]      
 5   secondary_description   string[pyarrow]      
 6   index_code              string[pyarrow]      
 7   primary_type            string[pyarrow]      
 8   description             string[pyarrow]      
 9   location_description    string[pyarrow]      
 10  arrest                  bool[pyarrow]        
 11  domestic                bool[pyarrow]        
 12  beat                    string[pyarrow]      
 13  district                string[pyarrow]      
 14  ward                    string[pyarrow]      
 15  community_area 

In [6]:
cols = ['year', 'date', 'arrest_date', 'race', 'fbi_code_desc', 'arrest', 'domestic', 'ward', 'community_area', 'beat', 'district', 'zip_code', 'zip_code_area', 
        'primary_neighborhood', 'secondary_neighborhood', 'neighborhood_area', 'time_of_day', 'month' , 'day_of_week', 'quarter', 'year_quarter'
       ]
# copy
df = df[cols].copy()

# re-index
df = df.reset_index(drop=True)

# collapse fragmented blocks internally
df._consolidate_inplace()

# display
df.head()

,year,date,arrest_date,race,fbi_code_desc,arrest,domestic,ward,community_area,beat,district,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,time_of_day,month,day_of_week,quarter,year_quarter
0,2025,2025-12-23 10:45:00,2025-12-23 10:45:00,WHITE,Drug Abuse Violations,True,False,28,27,1135,011,60612,106718949.397,Garfield Park,GARFIELD PARK,89976069.5947,Morning,December,Tuesday,Q4,2025-Q4
1,2025,2025-12-22 07:03:00,2025-12-22 07:12:00,BLACK,Miscellaneous Non-Index Offenses,True,False,42,8,1831,018,60654,15869961.5669,River North,RIVER NORTH,38766442.5194,Early Morning,December,Monday,Q4,2025-Q4
2,2025,2025-12-10 15:20:00,2025-12-10 17:32:00,BLACK,Simple Battery,True,False,12,59,912,009,60608,176505462.842,Mckinley Park,"BRIGHTON PARK,MCKINLEY PARK",39431799.6479,Afternoon,December,Wednesday,Q4,2025-Q4
3,2025,2025-11-21 04:50:00,2025-11-21 04:15:00,BLACK,Simple Battery,True,False,24,27,1134,011,60612,106718949.397,Garfield Park,GARFIELD PARK,89976069.5947,Early Morning,November,Friday,Q4,2025-Q4
4,2025,2025-11-17 06:20:00,2025-11-17 06:30:00,BLACK,Larceny – Theft,True,False,30,15,1731,017,60641,113903341.241,Portage Park,PORTAGE PARK,110196097.139,Early Morning,November,Monday,Q4,2025-Q4


## Check Data

### User Function(s)

In [7]:
def any_nans(data):
    # .sum() on PyArrow columns is very fast
    null_counts = data.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    
    if not null_counts.empty:
        print("NaNs with Columns Name and Count:")
        # Adding percentage helps put the 8.4M rows into perspective
        percent = (null_counts / len(data)) * 100
        out = pd.DataFrame({'Count': null_counts, 'Percentage': percent.round(4)})
        print(out)
    else:
        print("No NaNs found")

In [8]:
# check for nulls
df.isnull().values.any()

np.True_

In [9]:
# Find NaNs
any_nans(df)

NaNs with Columns Name and Count:
                          Count  Percentage
arrest_date             8084606     95.2688
race                    8084606     95.2688
zip_code_area            116470      1.3725
primary_neighborhood     116526      1.3731
secondary_neighborhood   116526      1.3731
neighborhood_area        116526      1.3731


In [10]:
# Summary statistics
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
year,8486097,25,2002,486823,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8486097,NaN,NaN,NaN,2011-08-08 23:50:45,2001-01-01 00:00:00,2005-06-16 19:00:00,2010-06-13 21:00:00,2017-05-29 19:00:00,2025-12-24 00:00:00,NaN
arrest_date,401491,NaN,NaN,NaN,2019-08-22 03:02:36,2014-01-01 00:02:00,2016-12-18 19:15:00,2019-04-21 04:48:00,2022-08-07 16:19:00,2025-12-28 20:47:00,NaN
race,401491,7,BLACK,287697,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fbi_code_desc,8486097,26,Larceny – Theft,1804200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrest,8486097,2,False,6339197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
domestic,8486097,2,False,7022389,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ward,8486097,51,0,614812,NaN,NaN,NaN,NaN,NaN,NaN,NaN
community_area,8486097,78,0,613755,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beat,8486097,305,421,65834,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Wrangle

In [11]:
# display number of unique values
for i in df.columns:
    print(f"{i}: {df[i].nunique():,}")

year: 25
date: 3,545,853
arrest_date: 371,755
race: 7
fbi_code_desc: 26
arrest: 2
domestic: 2
ward: 51
community_area: 78
beat: 305
district: 23
zip_code: 60
zip_code_area: 59
primary_neighborhood: 98
secondary_neighborhood: 78
neighborhood_area: 98
time_of_day: 6
month: 12
day_of_week: 7
quarter: 4
year_quarter: 100


In [ ]:
# 1. Create a boolean mask using the underlying Arrow arrays (fastest)
mask = df['primary_neighborhood'].str.lower() != df['secondary_neighborhood'].str.lower()

# 2. Slice, Drop Duplicates, and then Sort
# Reducing the rows BEFORE sorting is the key to speed.
diff_neighborhoods = (
    df.loc[mask, ['ward','community_area','zip_code','beat','district','primary_neighborhood','secondary_neighborhood']]
    .drop_duplicates()
    .sort_values('primary_neighborhood')
)

# display
print(diff_neighborhoods.to_string())

        ward community_area zip_code  beat district primary_neighborhood              secondary_neighborhood
305       33             14    60625  1713      017          Albany Park              NORTH PARK,ALBANY PARK
349       39             14    60630  1722      017          Albany Park              NORTH PARK,ALBANY PARK
360       39             14    60625  1722      017          Albany Park              NORTH PARK,ALBANY PARK
622       33             14    60625  1712      017          Albany Park              NORTH PARK,ALBANY PARK
738       35             14    60625  1723      017          Albany Park              NORTH PARK,ALBANY PARK
1053      39             14    60625  1712      017          Albany Park              NORTH PARK,ALBANY PARK
1343      33             14    60625  1723      017          Albany Park              NORTH PARK,ALBANY PARK
1623      39             14    60630  1712      017          Albany Park              NORTH PARK,ALBANY PARK
1920      33       

* Chicago crime data shows that incidents are distributed across wards, community areas, ZIP codes, and neighborhoods due to the city's overlapping geographic boundaries. Wards (50 political districts) are redrawn every decade for equal population representation and often split community areas and neighborhoods. Community areas (77 fixed zones since the 1920s) provide stable units for long-term analysis but do not align with fluid neighborhood perceptions or postal ZIP codes.

* Chicago Police beats and districts provide the most granular operational layer in crime data, with districts (22 citywide, numbered 001-022) covering broad sectors and beats (over 200 smaller zones, e.g., 1713) assigning specific officer patrols for community policing. In your Albany Park examples, all incidents fall under District 017 (17th, covering North Side neighborhoods like Albany Park) but vary across beats 1712-1724 due to precise geocoding within that district.
​
* Districts remain stable for your data (e.g., 017 for Albany Park ZIPs 60625/60630, 014/012 shifts for Wicker Park), while beats like 1712 (central Albany Park) or 1723 (near Lawrence/Foster) split neighborhoods for targeted response—explaining intra-neighborhood variations despite fixed community area 14. Zeros in ward/community indicate unknown values.

* Chicago has 77 community areas, a fixed geographic system created in the 1920s by University of Chicago researchers for stable statistical analysis. These areas enable consistent tracking of trends, such as crime rates, over decades, unaffected by political redistricting (wards) or postal changes (ZIP codes).



In [43]:
# Add leading zeros to ward & community_area (optimized for PyArrow)
cols = ['ward', 'community_area']
# iterate over columns
for col in cols:
    df[col] = df[col].astype('string[pyarrow]').str.zfill(2)

In [39]:
print(textwrap.fill(str(sorted(df['ward'].fillna("Unknown").unique())), width=80))

['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12',
'13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
'26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
'39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50']


In [40]:
print(textwrap.fill(str(sorted(df['community_area'].fillna("Unknown").unique())), width=80))

['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12',
'13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
'26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38',
'39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51',
'52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64',
'65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77']


In [ ]:
print(f"Missing Ward total: {(df.ward == '00').sum():,}")
print(f"Missing Community Area total: {(df.community_area == '00').sum():,}")
print(f"Missing Beat total: {(df.beat.sum():,}")

Missing Ward total: 614,812
Missing Community Area total: 613,755


In [25]:
print(textwrap.fill(str(sorted(df['primary_neighborhood'].fillna("Unknown").unique())), width=180))

['Albany Park', 'Andersonville', 'Archer Heights', 'Armour Square', 'Ashburn', 'Auburn Gresham', 'Austin', 'Avalon Park', 'Avondale', 'Belmont Cragin', 'Beverly', 'Boystown',
'Bridgeport', 'Brighton Park', 'Bucktown', 'Burnside', 'Calumet Heights', 'Chatham', 'Chicago Lawn', 'Chinatown', 'Clearing', 'Douglas', 'Dunning', 'East Side', 'East Village',
'Edgewater', 'Edison Park', 'Englewood', 'Fuller Park', 'Gage Park', 'Galewood', 'Garfield Park', 'Garfield Ridge', 'Gold Coast', 'Grand Boulevard', 'Grand Crossing', 'Grant Park',
'Greektown', 'Hegewisch', 'Hermosa', 'Humboldt Park', 'Hyde Park', 'Irving Park', 'Jackson Park', 'Jefferson Park', 'Kenwood', 'Lake View', 'Lincoln Park', 'Lincoln Square',
'Little Italy, UIC', 'Little Village', 'Logan Square', 'Loop', 'Lower West Side', 'Magnificent Mile', 'Mckinley Park', 'Millenium Park', 'Montclare', 'Morgan Park', 'Mount
Greenwood', 'Museum Campus', 'Near South Side', 'New City', 'North Center', 'North Lawndale', 'North Park', 'Norwood Park

https://www.chicagopolice.org/statistics-data/crime-statistics/

| Interval (Inclusive, Exclusive) | Mathematical Notation | Label        | Hours Included    |
|--------------------------------|----------------------|--------------|-------------------|
| 1st: 0 to 4                    | \([0, 4)\)          | Late Night   | 0, 1, 2, 3       |
| 2nd: 4 to 8                    | \([4, 8)\)          | Early Morning| 4, 5, 6, 7       |
| 3rd: 8 to 12                   | \([8, 12)\)         | Morning      | 8, 9, 10, 11     |
| 4th: 12 to 16                  | \([12, 16)\)        | Afternoon    | 12, 13, 14, 15   |
| 5th: 16 to 20                  | \([16, 20)\)        | Evening      | 16, 17, 18, 19   |
| 6th: 20 to 24                  | \([20, 24)\)        | Night        | 20, 21, 22, 23   |

In [ ]:
df_crime.iloc[[124860 , 2291302, 504157]]

In [ ]:
result.head()

In [ ]:
# df_crime[df_crime.primary_neighborhood == 'Chicago Lawn'].sample(20)

In [ ]:
df_crime.zip_code.nunique(), df_crime.primary_neighborhood.nunique(), df_crime.iucr.nunique()

In [ ]:
print(sorted(list(df_crime.zip_code.unique())))

In [ ]:
# remove features
cols = ['case_number', 'date', 'block', 'primary_type', 'description', 
        'location_description', 'updated_on', 'primary_neighborhood', 'secondary_neighborhood']

In [ ]:
# convert to category
cols = ['beat', 'district', 'ward', 'community_area', 'year', 'zip_code', 'p_district', 'p_sector', 'p_beat']

In [ ]:
crime_year = df_crime.groupby(['year', 'primary_type']).size().reset_index(name='count')
crime_year.head()

In [ ]:
# pivot table
pivot = (crime_year.pivot(index='primary_type', columns='year', values='count').sort_index())
# display
pivot.head()

In [ ]:
# Heatmap
plt.figure(figsize=(14,18))
sns.heatmap(pivot, cmap='YlOrRd', annot_kws={"size": 7}, annot=True, fmt='g')
plt.title("Crime Trends by Type and Year") 
plt.xlabel("Year") 
plt.ylabel("Crime Type") 
plt.show()

**Note:**
- Domestic violence has no reported data after 2001
- Ritualism has gaps in data
- Non-Criminal data is missing prior to 2015
- Human trafficking data is missing prior to 2013

In [ ]:
# remove columns
cols = ['RITUALISM', 'NON-CRIMINAL', 'DOMESTIC VIOLENCE']

In [ ]:
# aggregate data
aggregate = df_crime.groupby(['year','primary_type', 'primary_neighborhood']).size().unstack().fillna(0)
aggregate.head()

In [ ]:
sorted(list(aggregate.index))

In [ ]:
aggregate.loc[2025]

In [ ]:
aggregate.loc[2025].loc[['ROBBERY', 'HOMICIDE']]

In [ ]:
aggregate.loc[
    [(2024, 'ROBBERY'), (2024, 'HOMICIDE'),
     (2025, 'ROBBERY'), (2025, 'HOMICIDE')]
]


In [ ]:
aggregate.loc[(slice(2024, 2025), ['ROBBERY', 'HOMICIDE']), :]

In [ ]:
reset_agg = aggregate.reset_index()
reset_agg.head()

## New Data Set

In [ ]:
reset_agg.primary_type.unique()

In [ ]:
# rearrange data
neighborhood_cols = reset_agg.columns[2:]   # all neighborhood columns

df_long = reset_agg.melt(
    id_vars=['year', 'primary_type'],
    value_vars=neighborhood_cols,
    var_name='neighborhood',
    value_name='crime_count'
)
df_long.head()

#### Calculating the Z-score highlights specialized anomalies—places where a specific crime type is occurring at a rate far beyond what is statistically normal for the rest of the city.
- A Z-score above 3.0 is typically considered a significant outlier; seeing scores of 5.0 to 8.0 indicates extreme geographic concentration.

In [ ]:
# Calculate mean and standard deviation per year/type
stats = df_long.groupby(['year', 'primary_type'], observed=False)['crime_count'].agg(['mean', 'std']).fillna(0)

# Merge back and calculate Z-Score
df_long = df_long.merge(stats, on=['year', 'primary_type'])
df_long['z_score'] = (df_long['crime_count'] - df_long['mean']) / df_long['std']

# Find the most "Extreme" statistical outliers
extremes = df_long[df_long['year'] == 2025].sort_values('z_score', ascending=False)

In [ ]:
extremes[['neighborhood', 'primary_type', 'crime_count', 'z_score']].head(10)

In [ ]:
extremes.describe(include='all').T

### Descriptive & EDA (2024 & 2025)

In [ ]:
# take a look at 2024 & 2025
df_2025 = df_long.iloc[:, :4][df_long.year ==2025].reset_index(drop=True)
df_2024 = df_long.iloc[:, :4][df_long.year ==2024].reset_index(drop=True)

In [ ]:
df_2025.head()

In [ ]:
df_2025.year.unique(), df_2024.year.unique()

## 2025 ANOVA 

In [ ]:
# Normality test on the 'crime_count' column
normality_results = pg.normality(df_2025['crime_count'])

print("--- Normality Test Results ---")
print(normality_results)

- The data is extremely non-normal, and test homoscedasticity.

In [ ]:
# Testing if variance in 'crime_count' is equal across different primary_type & neighborhood
homogeneity_p_results = pg.homoscedasticity(data=df_2025, dv='crime_count', group='primary_type')
homogeneity_n_results = pg.homoscedasticity(data=df_2025, dv='crime_count', group='neighborhood')


print("\n--- Homogeneity of Variance Results ---\n")
print("Homogeneity result for Type of Crime:\n", homogeneity_p_results.to_string())
print("\nHomogeneity result for Neighborhood:\n", homogeneity_n_results.to_string())

In [ ]:
# Use Welch's ANOVA instead of regular ANOVA
welch_res_2025_p = pg.welch_anova(data=df_2025, dv='crime_count', between='primary_type')
welch_res_2025_n = pg.welch_anova(data=df_2025, dv='crime_count', between='neighborhood')
print(welch_res_2025_p.to_string())
print("\n", welch_res_2025_n.to_string())

- This is partial eta‑squared (np2), an effect size.
    - 0.01 = small
    - 0.06 = medium
    - 0.14 = large
    - 0.43 = extremely large

- Interpretation:
    - About 43% of all variance in crime_count is explained by crime type.
    - That is a huge effect.

In [ ]:
# Testing if variance in 'crime_count' is equal across different 'neighborhood'
homogeneity_results = pg.homoscedasticity(data=df_2025, dv='crime_count', group='neighborhood')

print("\n--- Homogeneity of Variance Results ---")
print(homogeneity_results)

In [ ]:
# Use Welch's ANOVA instead of regular ANOVA
welch_res_2025 = pg.welch_anova(data=df_2025, dv='crime_count', between='neighborhood')
welch_res_2025

#### Conclusion
- Comparing Homoscedasticity and  for primary_type, which represents crime types, and neighborhood for 2025 reported crimes  

In [ ]:
# The "Robust" T-test approach:
robust_crime = pg.pairwise_tests(
    data=df_2024_2025, 
    dv='crime_count', 
    between='neighborhood', 
    parametric=True, 
    correction=True # This handles the unequal variance (Welch's)
)

In [ ]:
robust_crime.info()
robust_crime.head()

In [ ]:
# The "Robust" T-test approach with :
robust_crime_bonf = pg.pairwise_tests(
    data=df_2024_2025, 
    dv='crime_count', 
    between='neighborhood', 
    parametric=True, 
    correction=True, # This handles the unequal variance (Welch's)
    padjust='bonf'   # Bonferroni correction
)
#  statistical method to prevent false positives (Type I errors) 
# when performing multiple hypothesis tests by adjusting the significance level, 
# making it harder to declare results significant

In [ ]:
robust_crime_bonf.info()

In [ ]:

robust_crime_bonf.BF10 = robust_crime_bonf.BF10.astype('float')
robust_crime_bonf.info()

In [ ]:
robust_crime.info()
robust_crime.BF10 = robust_crime.BF10.astype('float')
robust_crime.sort_values(by='BF10', ascending=False).head(10)

In [ ]:
robust_crime_bonf.BF10 = robust_crime_bonf.BF10.astype('float')
robust_crime_bonf.info()

In [ ]:
robust_crime_bonf.sort_values(by='BF10', ascending=False).head(10)

In [ ]:
# This tests Year, Neighborhood, primary_type
model = ols('crime_count ~ C(year) + C(neighborhood) + C(primary_type)', data=df_long).fit()
anova_results = anova_lm(model, typ=2)

In [ ]:
anova_results

In [ ]:
print(anova_results)

In [ ]:
# largest crime
df_long_filtered['primary_type'].value_counts().nlargest(5).index

In [ ]:
# largest crime
df_long_filtered['neighborhood'].value_counts().nlargest(5).index

In [ ]:
# Create a count table
counts = df_long_filtered.groupby(['neighborhood', 'primary_type']).size().unstack(fill_value=0)

# Find combinations with only 1 sample (Standard Deviation is impossible)
single_samples = (counts == 1).sum().sum()

# Find combinations with 0 samples (Mean/SD are NaN)
zero_samples = (counts == 0).sum().sum()

print(f"Groups with 1 sample: {single_samples}")
print(f"Groups with 0 samples: {zero_samples}")

In [ ]:
# 1. Filter out pairs that couldn't be calculated (NaN)
# 2. Filter for significant results (p < 0.05)
significant_results = posthocs.dropna(subset=['p-unc'])
significant_results = significant_results[significant_results['p-unc'] < 0.05]

print(significant_results.head().to_string())

In [ ]:
import pathlib

# This gets the directory you are currently working in
current_dir = pathlib.Path.cwd()
# pathlib.Path.cwd()
# # To go two levels up, just like your original code intended:
# grandparent_dir = current_dir.parent.parent.absolute()
current_dir.parent.parent.absolute()

In [ ]:
try:
    # Works in .py scripts
    root = pathlib.Path(__file__).parent.parent.absolute()
except NameError:
    # Works in Jupyter/Interactive shells
    root = pathlib.Path.cwd().parent.parent.absolute()

print(f"Project root is: {root}")

In [ ]:
# Fixed Two‑way ANOVA
aov_inter = pg.anova(
    data=df_long_filtered,
    dv='crime_count',
    between=['neighborhood', 'primary_type'],
    detailed=True
)

print(aov_inter)

In [ ]:
# Two‑way ANOVA with interaction
aov_inter = pg.anova(
    data=df_long_filtered,
    dv='crime_count',
    between=['neighborhood', 'primary_type'],
    detailed=True
)

print(aov_inter.to_string())